In [2]:
import numpy as np
from ete3 import Tree  # Updated from ete2 to ete3

def readInput():      
    with open('UPGMA_Input.txt') as f:
        matrix = [list(map(float, ln.split())) for ln in f]
    
    matrix = np.array(matrix, dtype=np.float64)  # Ensure proper NumPy array type
    length = len(matrix)
    
    print("Distance matrix:")
    print(matrix)
    print("\nNumber of sequences:", length, "\n")
    
    return matrix, length

def matrixMinimum(matrix, length):
    min_index_i, min_index_j = 0, 0
    minimum = float('inf')
    
    for i in range(length):
        for j in range(i + 1, length):  # Avoid checking diagonal
            if matrix[i][j] < minimum:
                minimum = matrix[i][j]
                min_index_i, min_index_j = i, j
    
    return min_index_i, min_index_j

def upgma(matrix, length, dictionary):
    leaves = ["S" + str(i + 1) for i in range(length)]
    num_clusters = length
    
    while length > 1:
        num_clusters += 1
        min_i, min_j = matrixMinimum(matrix, length)
        
        new_cluster = "S" + str(num_clusters)
        distance = matrix[min_i][min_j] / 2.0
        size = 1
        distance1 = distance2 = distance

        if leaves[min_i] in dictionary:
            size += dictionary[leaves[min_i]][4]
            distance1 = distance - max(dictionary[leaves[min_i]][0], dictionary[leaves[min_i]][2])

        if leaves[min_j] in dictionary:
            size += dictionary[leaves[min_j]][4]
            distance2 = distance - max(dictionary[leaves[min_j]][0], dictionary[leaves[min_j]][2])
        
        dictionary[new_cluster] = [distance1, leaves[min_i], distance2, leaves[min_j], size]

        new_row = [(matrix[i][min_i] + matrix[i][min_j]) / 2 for i in range(length)]
        new_row.append(0.0)

        matrix = np.vstack([matrix, new_row])
        new_col = np.append(new_row, [0.0]).reshape(-1, 1)
        matrix = np.hstack([matrix, new_col])

        matrix = np.delete(matrix, [min_i, min_j], axis=0)
        matrix = np.delete(matrix, [min_i, min_j], axis=1)

        del leaves[min(max(min_i, min_j), len(leaves)-1)]
        del leaves[min(min_i, min_j)]
        leaves.append(new_cluster)
        
        length = len(matrix)

    return "S" + str(num_clusters)

def printCluster(dictionary, finalCluster):
    stack = [finalCluster]
    result = []
    current_prev = None
    
    while stack:
        current = stack.pop()
        if isinstance(current, float):
            if isinstance(current_prev, float):
                result.pop()
                result.append(")")
            result.append(":" + str(current))
            result.append(",")
        elif current in dictionary:
            stack.append(dictionary[current][0])
            stack.append(dictionary[current][1])
            stack.append(dictionary[current][2])
            stack.append(dictionary[current][3])
            result.append("(")
        else:
            result.append(current)
        current_prev = current

    result.pop()
    result.append(")")
    return ''.join(result)

if __name__ == "__main__":
    matrix, length = readInput()
    dictionary = {}
    finalCluster = upgma(matrix, length, dictionary)
    result = printCluster(dictionary, finalCluster) + ";"
    
    tree = Tree(result)
    print("*******************************************************************************")
    print("UPGMA Resultant Clustering:\n")
    print(result)
    print("\n", tree)
    print("*******************************************************************************")


ModuleNotFoundError: No module named 'ete3'